# 03 · Clean a messy export
The other file in `data_samples/` is a realistic mess: `$` and commas in numbers, text like `N/A`, two date formats, impossible dates, category typos, duplicates. Your job as an analyst: **fix, drop, or quarantine** each problem, and say why.

If you have not made it yet: `python ../scripts/generate_sample_data.py --messy`

In [ ]:
import sys
sys.path.insert(0, "..")          # lets the notebook import common.py from python_practice/
from common import *              # load_operations, check, OUT, ...
import pandas as pd

df = load_operations()            # reads the dashboard API if it is running, else data_samples/operations_clean.csv

In [ ]:
raw = pd.read_csv(ensure_sample(messy=True), dtype=str, keep_default_na=False)   # dtype=str: nothing is silently converted
print(len(raw), "raw rows")
raw.sample(8, random_state=1)

## Profile the problems first
Count before you clean, so you can prove you fixed it.

In [ ]:
print("duplicate rows:", raw.duplicated().sum())
print("category spellings:", sorted(raw["category"].unique()))
print("status spellings: ", sorted(raw["status"].unique()))
print("revenue values that are not plain numbers:", (~raw["revenue"].str.fullmatch(r"-?\d+(\.\d+)?")).sum())

## Your turn
Build `clean`: a DataFrame with real dtypes (dates as datetime, money as float) and **no** missing required values, duplicate `fact_id`s or negative money. Tips: `str.strip()`, `str.replace(r"[$,]", "", regex=True)`, `pd.to_numeric(errors="coerce")`, `pd.to_datetime(errors="coerce")`, `drop_duplicates`, `dropna(subset=...)`.

Stuck? A worked solution is in `../solutions/03_clean_messy_data_solution.py`.

In [ ]:
clean = None

In [ ]:
REQ = ["fact_id", "record_date", "entity_id", "revenue", "operational_cost", "units_processed"]
if clean is None:
    print("(write your cleaning code above first)")
else:
    check("no missing required values", not clean[REQ].isna().any().any(), "dropna(subset=REQUIRED) after coercing")
    check("fact_id unique", clean["fact_id"].is_unique, "drop_duplicates(subset='fact_id')")
    check("no negative money", bool((clean["revenue"] >= 0).all() and (clean["operational_cost"] >= 0).all()), "filter them out or quarantine them")
    check("numeric dtypes", pd.api.types.is_numeric_dtype(clean["revenue"]) and pd.api.types.is_numeric_dtype(clean["operational_cost"]), "pd.to_numeric")
    check("real dates (not text)", bool(clean["record_date"].map(lambda v: hasattr(v, "year")).all()), "pd.to_datetime(..., errors='coerce'), then drop the NaT rows")
    print(f"kept {len(clean)} of {len(raw)} rows")

**Real-world note:** never silently throw rows away. Keep a `rejected` table with the reason for each, and report how many were dropped (the Airflow example fails the pipeline if too many are).